In [ ]:
!pip install ev2gym
!pip install pandapower
!pip install multicopula

# Setting up the stage - Markov Decision Processes

Let's introduce our guiding example, a *Vehicle-to-Grid* (V2G) management problem where a power supplier needs to charge electric vehicles (EVs) arriving randomly along the day, while optimizing its operational costs.

In [1]:
from ev2gym.models.ev2gym_env import EV2Gym
from ev2gym.baselines.mpc.V2GProfitMax import V2GProfitMaxOracle
from ev2gym.baselines.heuristics import ChargeAsFastAsPossible

config_file = "ev2gym/example_config_files/V2GProfitPlusLoads.yaml"

# Initialize the environment
env = EV2Gym(config_file=config_file)#,
              #save_replay=True,
              #save_plots=True)

/home/emmanuel/anaconda3/lib/python3.13/site-packages/ev2gym/utilities/loaders.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Let's take a peek into the config file.

In [2]:
from pygments import highlight
from pygments.formatters import Terminal256Formatter
from pygments.lexers import YamlLexer

config_file_contents = open(config_file).read()
print(highlight(config_file_contents, YamlLexer(), Terminal256Formatter()))

# This yml file is used to configure the evsim simulation

##############################################################################
# Simulation Parameters
##############################################################################
timescale: 15 # in minutes per step
simulation_length: 112 #90 # in steps per simulation

##############################################################################
# Date and Time
##############################################################################
# Year, month, 
year: 2022 # 2015-2023
month: 1 # 1-12
day: 17 # 1-31
# Whether to get a random date every time the environment is reset
random_day: True # True or False
random_hour: False # True or False

# Simulation Starting Time
# Hour and minute do not change after the environment has been reset
hour: 5 # Simulation starting hour (24 hour format)
minute: 0 # Simulation starting minute (0-59)

# Simulate weekdays, weekends, or both
simulation_days: weekdays # weekdays, weekends, or both

There are lots of lines in this configuration file. Let's summarize.  
We are simulating 112 time steps of 15-minutes length (that's 28 hours in total), on a random day in January 2022, starting at 5am. The scenario follows a "weekdays" profile and the vehicles arrive and leave as if the location is a "workplace".  
The distribution network we operate has a single transformer, 25 charging stations, each with a single EV charging port.  
The vehicles are heterogenous.

In [3]:
print(len(env.transformers))
print(len(env.charging_stations))
print(env.number_of_ports_per_cs)

1
25
1


Let's take a look at the initial state of our system.

In [4]:
state, _ = env.reset()

print("Connected EVs at each charging station")
for cs in env.charging_stations:
    print(cs.evs_connected)

print("\nState variables")
print(state)
print(len(state))
# "state" holds the following variables:
# current_step / simulation_length
# power_setpoint[current_step]
# power_usage[current_step-1]
# for each EV, [a,b,c] where 
#   a=b=c=0 if vehicle not connected, 
#   a=1 if vehicle fully charged, otherwise 0.5
#   b=total energy exchanged with vehicle
#   c=stay duration of vehicle
# So, overall, that is 3*(1+nb_vehicles) variables.

import numpy as np
print("\nA prettier display of state variables")
def reshape_state(env, s):
    return np.reshape(s, (1+env.cs*env.number_of_ports_per_cs,3))

print(reshape_state(env, state))

Connected EVs at each charging station
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]

State variables
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0.]
78

A prettier display of state variables
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


The decision variables for our problem are the current intensity, for each charging port, at every time step.

In [5]:
print(env.action_space) # one action variable per charging station (power output, between -1 and 1)

Box(-1.0, 1.0, (25,), float64)


Let's run a simulation.  
For this, we need a function that provides the decided current intensity at each time step. That's what we will call an *agent*.

In [5]:
#agent = V2GProfitMaxOracle(env)#,verbose=True) # optimal solution
#        or 
agent = ChargeAsFastAsPossible() # heuristic
for t in range(env.simulation_length):
    actions = agent.get_action(env) # get action from the agent
    new_state, _, done, truncated, stats = env.step(actions)  # takes action

In [6]:
stats

{'total_ev_served': np.int64(17),
 'total_profits': np.float64(-18.55070058777772),
 'total_energy_charged': np.float64(318.40147797417364),
 'total_energy_discharged': np.int64(0),
 'average_user_satisfaction': np.float64(0.999774936061381),
 'power_tracker_violation': np.float64(1273.6059118966948),
 'tracking_error': np.float64(47977.18516641437),
 'energy_tracking_error': np.float64(318.4014779741737),
 'energy_user_satisfaction': np.float64(99.97749360613811),
 'std_energy_user_satisfaction': np.float64(0.0900255754475719),
 'min_energy_user_satisfaction': np.float64(99.61739130434782),
 'total_steps_min_emergency_battery_capacity_violation': 0,
 'total_transformer_overload': np.float64(36.92766015302536),
 'battery_degradation': np.float64(0.0007613959913211008),
 'battery_degradation_calendar': np.float64(0.00032769255433816476),
 'battery_degradation_cycling': np.float64(0.00043370343698293613),
 'total_reward': np.float64(-47977.18516641437),
 'saved_grid_energy': 0,
 'voltage

Let's re-run that step by step, to see how the state changes. Run the second cell below repeatedly.

In [124]:
state, _ = env.reset()

In [162]:
action = agent.get_action(env)
new_state, _, done, truncated, stats = env.step(action)  # takes action
print(env.current_step)
print(action)
print(reshape_state(env, new_state))
#print(reward)

38
[0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 1. 1.
 1.]
[[ 0.33928571  0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 1.         25.53715751 22.        ]
 [ 0.          0.          0.        ]
 [ 1.         16.00607092 20.        ]
 [ 1.          7.97105886 22.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 1.         16.59793259 22.        ]
 [ 1.         22.42932031 18.        ]
 [ 1.          6.19150691 21.        ]
 [ 1.         10.2625936  18.        ]
 [ 1.         22.27702641 21.        ]
 [ 1.         30.61481791 25.        ]
 [ 1.         11.16934937 24.        ]
 [ 1.         15.31242157 18.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 1.         11.51798

In [165]:
env.charging_stations[3].evs_connected[0].time_of_departure

46

In [166]:
help(env)

Help on EV2Gym in module ev2gym.models.ev2gym_env object:

class EV2Gym(gymnasium.core.Env)
 |  EV2Gym(
 |      config_file=None,
 |      load_from_replay_path=None,
 |      replay_save_path='./replay/',
 |      generate_rnd_game=True,
 |      seed=None,
 |      save_replay=False,
 |      save_plots=False,
 |      state_function=<function PublicPST at 0x7fb8b3708040>,
 |      reward_function=<function SquaredTrackingErrorReward at 0x7fb8b36eb600>,
 |      cost_function=None,
 |      eval_mode='Normal',
 |      lightweight_plots=False,
 |      empty_ports_at_end_of_simulation=True,
 |      extra_sim_name=None,
 |      verbose=False,
 |      render_mode=None
 |  )
 |
 |  Method resolution order:
 |      EV2Gym
 |      gymnasium.core.Env
 |      typing.Generic
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      config_file=None,
 |      load_from_replay_path=None,
 |      replay_save_path='./replay/',
 |      generate_rnd_game=True,
 |      seed=Non

In [167]:
print(env.current_step)
print(env.power_setpoints[env.current_step])
print(env.power_setpoints)
print(env.current_power_usage)

38
0.0
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          9.9        17.93228415
 13.79256847 17.60196671 22.78571802 30.87702411 70.09268583 89.91940135
 92.75958821 75.06677674 94.8944744  85.46419923 76.71380507 68.29075628
 55.87302303 36.97078714 18.0921013  12.60541019  9.1225499   2.18812319
  0.53240053  0.10528942  0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.      

In [168]:
print(len(env.charging_stations))

25


In [1]:
from ev2gym.models.ev2gym_env import EV2Gym
from ev2gym.baselines.mpc.V2GProfitMax import V2GProfitMaxOracle
from ev2gym.baselines.heuristics import ChargeAsFastAsPossible
from ev2gym.rl_agent.state import V2G_profit_max

#config_file = "ev2gym/example_config_files/V2GProfitPlusLoads.yaml"
config_file = "custom.yaml"

# Initialize the environment
env = EV2Gym(config_file=config_file,
              save_replay=False,
              save_plots=False,
              state_function=V2G_profit_max)

/home/emmanuel/anaconda3/lib/python3.13/site-packages/ev2gym/utilities/loaders.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
state, _ = env.reset()
print("Connected EVs at each charging station")
for cs in env.charging_stations:
    print(cs.evs_connected)

# "state" holds the following variables:
# current_step
# power_usage[current_step-1]
# charge_prices for the next 20 time steps (padded with zeros at the end if needed)
# for each EV, [a,b,c] where 
#   state of charge (zero if no vehicle)
#   remaining time until departure (zero if no vehicle)
# So, overall, that is 2+20+2*nb_vehicles variables.
print("\nState variables")
print(state)
print(len(state))

Connected EVs at each charging station
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]
[None]

State variables
[0.      0.      0.41508 0.41508 0.41508 0.41508 0.43397 0.43397 0.43397
 0.43397 0.41619 0.41619 0.41619 0.41619 0.40495 0.40495 0.40495 0.40495
 0.3969  0.3969  0.3969  0.3969  0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.      0.      0.      0.
 0.      0.      0.      0.      0.      0.     ]
42


In [3]:
import time

agent = ChargeAsFastAsPossible() # heuristic
start = time.time()
for ep in range(100):
    print("episode", ep)
    env.reset()
    for t in range(env.simulation_length):
        actions = agent.get_action(env) # get action from the agent
        new_state, _, done, truncated, stats = env.step(actions)  # takes action

end = time.time()
print(end - start)

episode 0
episode 1
episode 2
episode 3
episode 4
episode 5
episode 6
episode 7
episode 8
episode 9
episode 10
episode 11
episode 12
episode 13
episode 14
episode 15
episode 16
episode 17
episode 18
episode 19
episode 20
episode 21
episode 22
episode 23
episode 24
episode 25
episode 26
episode 27
episode 28
episode 29
episode 30
episode 31
episode 32
episode 33
episode 34
episode 35
episode 36
episode 37
episode 38
episode 39
episode 40
episode 41
episode 42
episode 43
episode 44
episode 45
episode 46
episode 47
episode 48
episode 49
episode 50
episode 51
episode 52
episode 53
episode 54
episode 55
episode 56
episode 57
episode 58
episode 59
episode 60
episode 61
episode 62
episode 63
episode 64
episode 65
episode 66
episode 67
episode 68
episode 69
episode 70
episode 71
episode 72
episode 73
episode 74
episode 75
episode 76
episode 77
episode 78
episode 79
episode 80
episode 81
episode 82
episode 83
episode 84
episode 85
episode 86
episode 87
episode 88
episode 89
episode 90
episode 9